In [ ]:
# Libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.api.types import CategoricalDtype
from pathlib import Path
from functools import reduce

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Update data_path to reflect the mounted Google Drive path
data_path = "/content/drive/My Drive/STAT390 Data/All Calls by Month/"

In [ ]:
folder = Path(data_path)
files = sorted(list(folder.glob("*.csv")) + list(folder.glob("*.xlsx")))

### Reading first 5 rows of all data files
The code chunk below reads the first 5 rows of all data files. This is to check the columns that are present in all the data files.

In [ ]:
i=0; df = []
for f in files:
    if f.suffix.lower() == ".csv":
        df.append(pd.read_csv(f, header=0, dtype=str, engine="python", skip_blank_lines=False, nrows = 5))
    else:  # .xlsx
        df.append(pd.read_excel(f, sheet_name=0, header=0, dtype=str, nrows = 5))
    #df_main = pd.concat([df_main, df], ignore_index=True)
    print(i, f.stem, df[i].shape)
    i = i + 1

0 April 2024 (5, 63)
1 April 2025 (5, 63)
2 August 2024 (5, 57)
3 August 2025 (5, 63)
4 December 2024 (5, 55)
5 February 2025 (5, 63)
6 January 2025 (5, 64)
7 July 2024 (5, 55)
8 July 2025 (5, 63)
9 June 2024 (5, 63)
10 June 2025 (5, 63)
11 March 2025 (5, 64)
12 May 2024 (5, 63)
13 May 2025 (5, 63)
14 November 2024 (5, 63)
15 October 2024 (5, 55)
16 September 2024 (5, 55)
17 September 2025 (5, 69)


The code chunk below identifies the columns missing in at least one DataFrame.

In [ ]:
all_cols = reduce(lambda x, y: x | set(y.columns), df[1:], set(df[0].columns))
common_cols = reduce(lambda x, y: x & set(y.columns), df[1:], set(df[0].columns))
common_cols
not_in_all = all_cols - common_cols
print("Columns missing from at least one dataframe:", not_in_all)

Columns missing from at least one dataframe: {'Call Recording Platform Name', 'Call Recording Trigger', 'Auto Attendant Key Pressed', 'Device owner UUID', 'Answered Elsewhere', 'User', 'Redirecting party UUID', 'Public Calling IP Address', 'Recall Type', 'Queue Type', 'Public Called IP Address', 'Hold Duration', 'Column1', 'PSTN vendor name2', 'Call Recording Result', 'External caller ID number', 'Original called party UUID'}


The code chunk below prints the columns present in all the data files.

In [ ]:
print(common_cols)

{'Call type', 'Location', 'Report time', 'Redirect reason', 'Network call ID', 'Local SessionID', 'Ring duration', 'Site timezone', 'Call transfer time', 'Authorization code', 'Duration', 'User type', 'Report ID', 'OS type', 'Remote SessionID', 'Remote call ID', 'User number', 'Answer Indicator', 'Redirecting number', 'Client version', 'Final local sessionID', 'Model', 'Final remote sessionID', 'Device Mac', 'Release time', 'User UUID', 'Answered', 'PSTN provider ID', 'Org UUID', 'Route group', 'Outbound trunk', 'Department ID', 'Releasing party', 'Correlation ID', 'Start time', 'Related call ID', 'Site UUID', 'Transfer related call ID', 'International Country', 'Local call ID', 'Call outcome reason', 'PSTN vendor Org ID', 'Inbound trunk', 'Call ID', 'Related reason', 'Called number', 'Call outcome', 'PSTN legal entity', 'Site main number', 'Client type', 'PSTN vendor name', 'Sub client type', 'Original reason', 'Answer time', 'Direction'}


### Reading all the data files
All the datafiles are read with the common columns read first.

In [ ]:
df_main = pd.DataFrame(columns=list(common_cols))

In [ ]:
i=0;
for f in files:
    if f.suffix.lower() == ".csv":
        df = pd.read_csv(f, header=0, dtype=str, engine="python", skip_blank_lines=False)
    else:  # .xlsx
        df = pd.read_excel(f, sheet_name=0, header=0, dtype=str)
    df_main = pd.concat([df_main, df], ignore_index=True)
    print(i, f.stem, df.shape)
    i = i + 1

0 April 2024 (56662, 63)
1 April 2025 (63636, 63)
2 August 2024 (63262, 57)
3 August 2025 (57071, 63)
4 December 2024 (49445, 55)
5 February 2025 (63669, 63)
6 January 2025 (62623, 64)
7 July 2024 (62292, 55)
8 July 2025 (60438, 63)
9 June 2024 (56763, 63)
10 June 2025 (54598, 63)
11 March 2025 (59149, 64)
12 May 2024 (62944, 63)
13 May 2025 (55428, 63)
14 November 2024 (49953, 63)
15 October 2024 (62354, 55)
16 September 2024 (61250, 55)
17 September 2025 (56571, 69)


### Converting date to datetime format

In [ ]:
df_main["Start time"] = pd.to_datetime(df_main["Start time"], utc=True)

In [ ]:
df_main["Start time"] = df_main["Start time"].dt.tz_convert("America/Chicago").dt.tz_localize(None)

In [ ]:
df_main["Start time"].head()

,Start time
0,2024-04-30 18:58:53.988
1,2024-04-30 18:56:37.386
2,2024-04-30 18:54:59.099
3,2024-04-30 18:54:59.099
4,2024-04-30 18:54:52.336


In [ ]:
df_allcallsdata = df_main.copy()

# Updated Steps

## Step 1: Group by "Correlation ID" and "Start time"

In [ ]:
df_allcallsdata.sort_values(by=['Correlation ID', 'Start time'], ascending=[True, True])[['Correlation ID', 'Start time', 'Called number', 'Duration']].head(50)

,Correlation ID,Start time,Called number,Duration
600224,00000fe9-dfa0-41c5-986b-d9ce731f2715,2025-06-27 11:25:23.791,13123411070,4
929258,00001e73-ce33-48f1-b531-bddc7ee3965d,2024-10-05 12:30:09.903,13124312299,1961
267542,00006ccc-8250-4993-90c0-bbfd41f7dd24,2024-12-11 14:41:47.355,13123478311,44
267540,00006ccc-8250-4993-90c0-bbfd41f7dd24,2024-12-11 14:42:05.358,13123478300,44
267541,00006ccc-8250-4993-90c0-bbfd41f7dd24,2024-12-11 14:42:05.358,13123478300,44
818656,00008bec-c404-48cc-8276-1f276bf4229a,2025-05-06 13:04:29.746,13122296344,65
818654,00008bec-c404-48cc-8276-1f276bf4229a,2025-05-06 13:04:29.749,13123478300,65
818655,00008bec-c404-48cc-8276-1f276bf4229a,2025-05-06 13:04:29.749,13123478300,65
974134,000090ae-a71c-49e9-99d6-fdc7078dfa48,2024-09-13 14:35:32.428,17086568223,57
828444,0000aef0-6f35-4560-a4be-5b3bc31fbefb,2024-11-28 09:49:47.564,13123411070,2


## Step 2: Extract inbound calls

In [ ]:
### Remove call duration = 0
# convert the "Duration" column into numeric
df_allcallsdata["Duration"] = pd.to_numeric(df_allcallsdata["Duration"], errors="coerce")
df_allcallsdata = df_allcallsdata[df_allcallsdata["Duration"] > 0]


### Sort by Correlation ID and Start Time (chronological order)
df_allcallsdata = df_allcallsdata.sort_values(by=["Correlation ID", "Start time"])


### Classify call type (Inbound/Outbound/Internal)
def classify_call(row):
    if row["PSTN vendor name"] == "CallTower" and row["Direction"] == "ORIGINATING":
        return "Outbound"
    elif row["PSTN vendor name"] == "CallTower" and row["Direction"] == "TERMINATING":
        return "Inbound"
    elif row["PSTN vendor name"] == "NA":
        return "Internal"
    else:
        return "Other"

# Temporary column for per-row classification
df_allcallsdata["TempCallType"] = df_allcallsdata.apply(classify_call, axis=1)

# Propagate earliest-leg call type to all legs of the same call
earliest_calltype = df_allcallsdata.groupby("Correlation ID").first().reset_index()[["Correlation ID", "TempCallType"]]
df_allcallsdata = df_allcallsdata.drop(columns=["TempCallType"])
df_allcallsdata = df_allcallsdata.merge(earliest_calltype.rename(columns={"TempCallType":"Inbound/Outbound"}),
                                        on="Correlation ID", how="left")

In [ ]:
# Keep only the inbound calls
df_allcallsdata_inbound = df_allcallsdata[df_allcallsdata["Inbound/Outbound"] == "Inbound"]

In [ ]:
# Get a general sense of the proportion of data kept
print('Number of observations in the original dataset:', df_allcallsdata.shape[0])
print('Number of observations in the inbound call dataset:', df_allcallsdata_inbound.shape[0])
print('Proportion of observations kept:', df_allcallsdata_inbound.shape[0] / df_allcallsdata.shape[0])

Number of observations in the original dataset: 1003089
Number of observations in the inbound call dataset: 757494
Proportion of observations kept: 0.755161306723531


In [ ]:
# Extract time features
df_allcallsdata_inbound["Start time"] = pd.to_datetime(df_allcallsdata_inbound["Start time"])
df_allcallsdata_inbound["Hour"] = df_allcallsdata_inbound["Start time"].dt.hour
df_allcallsdata_inbound["DayOfWeek"] = df_allcallsdata_inbound["Start time"].dt.weekday + 1  # Monday=1, Sunday=7
df_allcallsdata_inbound["Month"] = df_allcallsdata_inbound["Start time"].dt.month
df_allcallsdata_inbound["Quarter"] = df_allcallsdata_inbound["Start time"].dt.quarter
df_allcallsdata_inbound["Year"] = df_allcallsdata_inbound["Start time"].dt.year

/tmp/ipython-input-2771028058.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_allcallsdata_inbound["Start time"] = pd.to_datetime(df_allcallsdata_inbound["Start time"])
/tmp/ipython-input-2771028058.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_allcallsdata_inbound["Hour"] = df_allcallsdata_inbound["Start time"].dt.hour
/tmp/ipython-input-2771028058.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = valu

## Step 3: Add a column to describes the numbers' functions

In [ ]:
number_map = {
    # From "QuestionsForCynthia_30Sep_Krish"
    "13123478300": "Internal voicemail - not client related",
    "13123411070": "Main number",
    "13124235938": "Community Legal Clinics",
    "13122296300": "Direct Line to Front Desk",
    "1180": "Transfers to English Queue Options (Legal Menu)",
    "13125068646": "Transfers to the English main menu",
    "13124312299": "Farmworker main number / Migrant Legal Assistance Program",
    "13122296079": "Nursing Home Ombudsman",
    "13122296344": "Bankruptcy Helpdesk Voicemail",
    "13122296071": "Criminal Records",
    "13125068647": "Transfers to the Spanish main menu",
    "13123478340": "Veterans Rights Project Voicemail",
    "13122296014": "Markham Eviction Help Desk",
    "13123478309": "HIV Intake Voicemail",
    "13122296072": "Juvenile Expungement Help Desk (JEHD)",
    "13124235904": "Austin Intake Voicemail",
    "2302": "Staff Directory English Transfer",

    # From "Intake phone numbers for reporting"
    "13123478347": "A2J Immigration (Lisa Palumbo's direct line)",
    "18882652188": "A2J Immigration (Lisa Palumbo's direct line)",  # rings to 13123478347
    "13124235900": "CLASP Voicemail",
    "13123478392": "Education Law Referrals Voicemail",
    "13124235909": "Fair Housing Intake Voicemail",
    "13124312101": "OP Appeals Project",
    "13122296073": "Trafficking Survivors Assistance Project (TSAP)",
    "18004459025": "Migrant Legal Assistance Program",  # rings to 13124312299
    "18884018200": "Nursing Home Ombudsman"  # rings to 13122296079
}


In [ ]:
# Add the number description column for better visualization
df_allcallsdata_inbound["Number Description"] = df_allcallsdata_inbound["Called number"].astype(str).map(number_map)

# Replace unmapped numbers with the original 'Called number'
df_allcallsdata_inbound["Number Description"] = df_allcallsdata_inbound["Number Description"].fillna(df_allcallsdata_inbound["Called number"].astype(str))

/tmp/ipython-input-2595277813.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_allcallsdata_inbound["Number Description"] = df_allcallsdata_inbound["Called number"].astype(str).map(number_map)
/tmp/ipython-input-2595277813.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_allcallsdata_inbound["Number Description"] = df_allcallsdata_inbound["Number Description"].fillna(df_allcallsdata_inbound["Called number"].astype(str))


In [ ]:
df_allcallsdata_inbound["Number Description"].unique()

array(['Main number',
       'Farmworker main number / Migrant Legal Assistance Program',
       '13123478311', 'Internal voicemail - not client related',
       'Bankruptcy Helpdesk Voicemail', '13122296306',
       'Community Legal Clinics', '13125068649', '13122296029',
       'Staff Directory English Transfer', '13122296060',
       'Austin Intake Voicemail', 'Criminal Records',
       'Direct Line to Front Desk', '13123478312', '13122296346',
       '13122296078', '13122296365', 'Nursing Home Ombudsman',
       '13124312151', '13123478394', '13123478342', '13122296065',
       '13123478379', '13122296364', '13123478398', '2303', '13122296350',
       '13122296397', '13124235920', '13122296354', '13123478302',
       '13122296339', 'Veterans Rights Project Voicemail', '13123478329',
       'Transfers to English Queue Options (Legal Menu)',
       'Transfers to the English main menu', '13122296321', '13122296057',
       '13123478391', '13122296094', '13122296315', '13122296308',
  

In [ ]:
df_allcallsdata_inbound["Number Description"].nunique()

394

In [ ]:
df_allcallsdata_inbound["Number Description"].value_counts()

,count
Number Description,
Internal voicemail - not client related,222215
Main number,211793
Staff Directory English Transfer,55266
Community Legal Clinics,52411
Direct Line to Front Desk,34026
...,...
13122296398,2
13122616790,2
12406089897,1


## Find inbound calls with the journey: Main -> Front Desk -> Main

In [ ]:
# 1. Sort by ID and Time to ensure the flow is chronological
df_sorted = df_allcallsdata_inbound.sort_values(by=['Correlation ID', 'Start time'])

# 2. Create 'shifted' series to look ahead at the next 2 steps within the same ID
# shift(-1) gets the next row's value; shift(-2) gets the value 2 rows ahead
grouped_desc = df_sorted.groupby('Correlation ID')['Number Description']
next_desc = grouped_desc.shift(-1)
next_next_desc = grouped_desc.shift(-2)

# 3. Define the specific pattern: Main -> Front Desk -> Main
pattern_mask = (
    (df_sorted['Number Description'] == "Main number") &
    (next_desc == "Direct Line to Front Desk") &
    (next_next_desc == "Main number")
)

# 4. Extract the IDs that match this start point
target_ids = df_sorted.loc[pattern_mask, 'Correlation ID'].unique()

print(f"Found {len(target_ids)} Correlation IDs with this pattern.")
print(target_ids)

Found 0 Correlation IDs with this pattern.
[]


## Step 4: Map out the call flow for each call (for drill-down feature)

In [ ]:
# Sort properly
df_allcallsdata_inbound = df_allcallsdata_inbound.sort_values(
    ["Correlation ID", "Start time"]
)

# Remove consecutive duplicates within each call
df_nocons = (
    df_allcallsdata_inbound
    .assign(
        prev_desc=lambda df: df.groupby("Correlation ID")["Number Description"].shift(),
        prev_num=lambda df: df.groupby("Correlation ID")["Called number"].shift()
    )
    .loc[
        lambda df: ~(
            (df["Number Description"] == df["prev_desc"]) &
            (df["Called number"] == df["prev_num"])
        )
    ]
    .drop(columns=["prev_desc", "prev_num"])
)

# Now group into call flows
call_sequences = (
    df_nocons.groupby("Correlation ID")["Number Description"]
    .apply(lambda x: x.tolist())
    .reset_index(name="Call Flow")
)

# Add num_legs
call_sequences["num_legs"] = call_sequences["Call Flow"].apply(len)

# Optional: remove outliers
call_sequences_clean = call_sequences[call_sequences["num_legs"] <= 4].copy()

# Expand horizontally
max_legs = call_sequences_clean["Call Flow"].apply(len).max()
for i in range(max_legs):
    call_sequences_clean[f"Number Description_{i+1}"] = call_sequences_clean["Call Flow"].apply(
        lambda x: x[i] if len(x) > i else None
    )

In [ ]:
# Preserve only the calls with number of legs smaller or equal to 4 since they comprise most of the calls

call_sequences["num_legs"].value_counts().sort_index()

,count
num_legs,
1,171428
2,97034
3,31842
4,16025
5,264
6,813
7,243
8,18
9,38


In [ ]:
print("The percentage of the calls preserved:", (171428+97034+31842+16025) / call_sequences.shape[0])

The percentage of the calls preserved: 0.9955843289313006


In [ ]:
# Write an ending label in the correct Number Description column that says on which transfer the call ended
# Helper function to convert numbers to ordinal strings
def ordinal(n):
    return "%d%s" % (n, "tsnrhtdd"[(n//10%10!=1)*(n%10<4)*n%10::4])

for idx, row in call_sequences_clean.iterrows():
    num_legs = row["num_legs"]

    if num_legs == 1:
        # Case: Intake only
        call_sequences_clean.at[idx, "Number Description_2"] = "Ended"

    else:
        # num_legs = N → ended on the (N-1)th transfer
        transfer_num = num_legs - 1
        ord_str = ordinal(transfer_num)

        end_label = f"Ended on the {ord_str} transfer"
        col_name = f"Number Description_{num_legs + 1}"

        call_sequences_clean.at[idx, col_name] = end_label

## Step 5: Understand the call distribution among the called numbers

In [ ]:
# Preview result
call_sequences_clean.head(20)

,Correlation ID,Call Flow,num_legs,Number Description_1,Number Description_2,Number Description_3,Number Description_4,Number Description_5
0,00000fe9-dfa0-41c5-986b-d9ce731f2715,[Main number],1,Main number,Ended,None,None,NaN
1,00001e73-ce33-48f1-b531-bddc7ee3965d,[Farmworker main number / Migrant Legal Assist...,1,Farmworker main number / Migrant Legal Assista...,Ended,None,None,NaN
2,00006ccc-8250-4993-90c0-bbfd41f7dd24,"[13123478311, Internal voicemail - not client ...",2,13123478311,Internal voicemail - not client related,Ended on the 1st transfer,None,NaN
3,00008bec-c404-48cc-8276-1f276bf4229a,"[Bankruptcy Helpdesk Voicemail, Internal voice...",2,Bankruptcy Helpdesk Voicemail,Internal voicemail - not client related,Ended on the 1st transfer,None,NaN
4,0000aef0-6f35-4560-a4be-5b3bc31fbefb,[Main number],1,Main number,Ended,None,None,NaN
5,0000ff52-cebd-430b-bca8-b8ae9a8b6e04,[Main number],1,Main number,Ended,None,None,NaN
6,0001424d-5229-46a5-9661-04dbc11e5273,"[13122296306, Internal voicemail - not client ...",2,13122296306,Internal voicemail - not client related,Ended on the 1st transfer,None,NaN
7,0001736f-cdab-4164-8756-6784d802f051,"[Main number, Community Legal Clinics, 1312506...",3,Main number,Community Legal Clinics,13125068649,Ended on the 2nd transfer,NaN
8,000197f8-fafe-46d3-a830-94aefc90fd90,[Internal voicemail - not client related],1,Internal voicemail - not client related,Ended,None,None,NaN
9,0001af3f-5951-4d02-bf61-27cab9793b9d,"[13122296029, Internal voicemail - not client ...",2,13122296029,Internal voicemail - not client related,Ended on the 1st transfer,None,NaN


In [ ]:
# Compare the datasize with original dataset
print("The shape for original dataset:", df_allcallsdata.shape)
print("The shape for processed data, with each row representing the flow for a inbound call:", call_sequences_clean.shape)
pct_all = call_sequences_clean.shape[0] / df_allcallsdata["Correlation ID"].nunique() * 100
print(f"The percentage of all calls kept: {pct_all:.2f}%")
pct_inbound = call_sequences_clean.shape[0] / df_allcallsdata_inbound["Correlation ID"].nunique() * 100
print(f"The percentage of inbound calls kept: {pct_inbound:.2f}%")

The shape for original dataset: (1003089, 73)
The shape for processed data, with each row representing the flow for a inbound call: (316329, 8)
The percentage of all calls kept: 65.23%
The percentage of inbound calls kept: 99.56%


In [ ]:
# Examine the proportion for each called number in the first step
call_sequences_clean['Number Description_1'].value_counts()

,count
Number Description_1,
Main number,208103
Farmworker main number / Migrant Legal Assistance Program,9620
Bankruptcy Helpdesk Voicemail,6641
Nursing Home Ombudsman,6457
Community Legal Clinics,3592
...,...
13123478305,1
13124235925,1
13122296010,1


In [ ]:
# Examine the proportion for each called number during the first transfe
call_sequences_clean['Number Description_2'].value_counts()

,count
Number Description_2,
Ended,171428
Internal voicemail - not client related,69657
Staff Directory English Transfer,27316
Community Legal Clinics,23401
Direct Line to Front Desk,13914
...,...
13122296368,1
13123478373,1
13123478353,1


In [ ]:
call_sequences_clean['Number Description_3'].value_counts()

,count
Number Description_3,
Ended on the 1st transfer,97034
Internal voicemail - not client related,27451
Transfers to English Queue Options (Legal Menu),4138
1182,1164
13125068649,677
...,...
13122616781,1
13124506726,1
13122296010,1


In [ ]:
call_sequences_clean['Number Description_4'].value_counts()

,count
Number Description_4,
Ended on the 2nd transfer,31842
Internal voicemail - not client related,10323
Transfers to the English main menu,3870
Transfers to the Spanish main menu,1083
13127536356,268
13125068649,267
Bankruptcy Helpdesk Voicemail,104
13127536357,81
17736766359,5


In [ ]:
call_sequences_clean['Number Description_5'].value_counts()

,count
Number Description_5,
Ended on the 3rd transfer,16025


## Step 6: Merge with the meaningful columns from the original dataset

In [ ]:
df_allcallsdata_inbound.columns

Index(['Call type', 'Location', 'Report time', 'Redirect reason',
       'Network call ID', 'Local SessionID', 'Ring duration', 'Site timezone',
       'Call transfer time', 'Authorization code', 'Duration', 'User type',
       'Report ID', 'OS type', 'Remote SessionID', 'Remote call ID',
       'User number', 'Answer Indicator', 'Redirecting number',
       'Client version', 'Final local sessionID', 'Model',
       'Final remote sessionID', 'Device Mac', 'Release time', 'User UUID',
       'Answered', 'PSTN provider ID', 'Org UUID', 'Route group',
       'Outbound trunk', 'Department ID', 'Releasing party', 'Correlation ID',
       'Start time', 'Related call ID', 'Site UUID',
       'Transfer related call ID', 'International Country', 'Local call ID',
       'Call outcome reason', 'PSTN vendor Org ID', 'Inbound trunk', 'Call ID',
       'Related reason', 'Called number', 'Call outcome', 'PSTN legal entity',
       'Site main number', 'Client type', 'PSTN vendor name',
       'Sub cli

In [ ]:
# Extract first leg info for each call
first_leg_info = df_allcallsdata_inbound.groupby("Correlation ID").first().reset_index()

# Select only the columns I want to merge
first_leg_info = first_leg_info[["Correlation ID", "Hour", "DayOfWeek", "Month", "Quarter", "Year", "Start time"]]

# Step 4: Merge with call_sequences_clean
call_sequences_merged = call_sequences_clean.merge(first_leg_info, on="Correlation ID", how="left")

# Preview
call_sequences_merged.head()


,Correlation ID,Call Flow,num_legs,Number Description_1,Number Description_2,Number Description_3,Number Description_4,Number Description_5,Hour,DayOfWeek,Month,Quarter,Year,Start time
0,00000fe9-dfa0-41c5-986b-d9ce731f2715,[Main number],1,Main number,Ended,None,None,NaN,11,5,6,2,2025,2025-06-27 11:25:23.791
1,00001e73-ce33-48f1-b531-bddc7ee3965d,[Farmworker main number / Migrant Legal Assist...,1,Farmworker main number / Migrant Legal Assista...,Ended,None,None,NaN,12,6,10,4,2024,2024-10-05 12:30:09.903
2,00006ccc-8250-4993-90c0-bbfd41f7dd24,"[13123478311, Internal voicemail - not client ...",2,13123478311,Internal voicemail - not client related,Ended on the 1st transfer,None,NaN,14,3,12,4,2024,2024-12-11 14:41:47.355
3,00008bec-c404-48cc-8276-1f276bf4229a,"[Bankruptcy Helpdesk Voicemail, Internal voice...",2,Bankruptcy Helpdesk Voicemail,Internal voicemail - not client related,Ended on the 1st transfer,None,NaN,13,2,5,2,2025,2025-05-06 13:04:29.746
4,0000aef0-6f35-4560-a4be-5b3bc31fbefb,[Main number],1,Main number,Ended,None,None,NaN,9,4,11,4,2024,2024-11-28 09:49:47.564


# Export the dataset

In [ ]:
inbound_call_workflow = call_sequences_merged

In [ ]:
inbound_call_workflow.to_csv("Dec6_AllCallsData_Inbound Call Workflow.csv", index=False)